# 03 -- Build the venue map layers

Produces the two venue layers the web app draws, directly into `src/data/venues/`:

1. **Venue centroids** (`venues-centroids.geo.json`): one point per TAC venue, computed from the venue boundary polygons so the two files can never drift apart.
2. **All arts locations** (`current_toronto_arts_locations_eddit.geo.json`): the ~2,600 point inventory of arts locations (the grey "All" dots on the map), converted from the TAC-maintained CSV.



---

| | |
|---|---|
| **Inputs** | `src/data/venues/venues-boundaries.geo.json` -- hand-edited venue polygons (reference, **not** generated)<br>`data/venues/tac-list/current_toronto_arts_locations_eddit.csv` -- location inventory maintained by TAC (`lat`, `lon`, funding columns) |
| **Outputs** | **`src/data/venues/venues-centroids.geo.json`**<br>**`src/data/venues/current_toronto_arts_locations_eddit.geo.json`** |
| **Run after** | Venue boundaries are up to date (`01_geometrize_tac_list.ipynb` + QGIS editing). |

To add or change a venue, edit `venues-boundaries.geo.json` (in QGIS) and re-run this notebook, then `02_geohash9_venues.ipynb`.


In [ ]:
import json
from pathlib import Path

import geopandas as gpd
import pandas as pd
from shapely.geometry import shape, mapping

# Repo root = nearest parent folder containing package.json, so this notebook runs from any depth.
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "package.json").exists())
DATA = ROOT / "data"                 # raw + intermediate files
SRC_DATA = ROOT / "src" / "data"     # final outputs consumed directly by the web app

BOUNDARIES_PATH = SRC_DATA / "venues" / "venues-boundaries.geo.json"
ARTS_CSV = DATA / "venues" / "tac-list" / "current_toronto_arts_locations_eddit.csv"

CENTROIDS_PATH = SRC_DATA / "venues" / "venues-centroids.geo.json"
ARTS_PATH = SRC_DATA / "venues" / "current_toronto_arts_locations_eddit.geo.json"

METRIC_CRS = "EPSG:32617"   # UTM 17N: centroids are computed in metres, then converted back to lon/lat

## Helper: write GeoJSON

Writes GeoJSON with one feature per line (the layout QGIS/GDAL produces) so that git diffs of these files stay readable. Properties are copied verbatim, so list-valued fields such as `disciplines_supported` are preserved.

In [ ]:
def _fmt(v):
    if isinstance(v, dict):
        return "{ " + ", ".join(f"{json.dumps(k)}: {_fmt(x)}" for k, x in v.items()) + " }"
    if isinstance(v, list):
        return "[ " + ", ".join(_fmt(x) for x in v) + " ]"
    return json.dumps(v, ensure_ascii=False)


def write_geojson(path, name, features):
    header = [
        "{",
        '"type": "FeatureCollection",',
        f'"name": {json.dumps(name)},',
        '"crs": { "type": "name", "properties": { "name": "urn:ogc:def:crs:OGC:1.3:CRS84" } },',
        '"features": [',
    ]
    body = ",\n".join(_fmt(f) for f in features)
    Path(path).write_text("\n".join(header) + "\n" + body + "\n]\n}\n", encoding="utf-8")

## 1. Venue centroids

Each venue's centroid is the centroid of its boundary polygon (computed in UTM so it is not distorted by latitude). Venue properties (`id`, `venue_name`, `disciplines_supported`, ...) are carried over unchanged.

In [ ]:
boundaries = json.loads(BOUNDARIES_PATH.read_text(encoding="utf-8"))

polygons = gpd.GeoSeries([shape(f["geometry"]) for f in boundaries["features"]], crs="EPSG:4326")
centroids = polygons.to_crs(METRIC_CRS).centroid.to_crs("EPSG:4326")

centroid_features = [
    {"type": "Feature", "properties": f["properties"], "geometry": {"type": "Point", "coordinates": [pt.x, pt.y]}}
    for f, pt in zip(boundaries["features"], centroids)
]
write_geojson(CENTROIDS_PATH, "venues-centroids-merged", centroid_features)
print(f"Wrote {len(centroid_features)} venue centroids to {CENTROIDS_PATH}")

## 2. All arts locations

Converts the TAC location inventory (CSV with `lat`/`lon`) to point GeoJSON. `TAC_funded_activities` becomes a true/false boolean; the other columns are kept as they are.

In [ ]:
arts = pd.read_csv(ARTS_CSV)
arts = arts.dropna(subset=["lat", "lon"])
arts["TAC_funded_activities"] = arts["TAC_funded_activities"].astype(str).str.upper().eq("TRUE")

arts_features = [
    {
        "type": "Feature",
        "properties": {k: (None if pd.isna(v) else v) for k, v in row.items()},
        "geometry": {"type": "Point", "coordinates": [row["lon"], row["lat"]]},
    }
    for row in arts.to_dict("records")
]
write_geojson(ARTS_PATH, "current_toronto_arts_locations_eddit", arts_features)
print(f"Wrote {len(arts_features)} arts locations to {ARTS_PATH}")